# 05.12 - Dimensionality Reduction (PCA)

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Principal Component Analysis (PCA) reduces the number of features by projecting data onto the directions of maximum variance (principal components).

## 2. Why Does This Matter?

PCA reduces dimensionality, speeds up training, removes redundancy, and enables visualization. It's used everywhere from image compression to exploratory analysis.

## 3. Prerequisites

- Phase 02 (Math - eigenvectors, variance)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain eigenvectors and variance
- Explain projection
- Implement PCA from scratch
- Use sklearn's PCA

## 5. Mental Model

PCA finds the directions (principal components) of maximum variance:

1. Center the data.
2. Compute the covariance matrix.
3. Find its eigenvectors (directions) and eigenvalues (variance).
4. Project data onto the top-k eigenvectors.

The first component captures the most variance.


## 6. Generate Data

Create correlated high-dimensional data.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 500
# Correlated features
base = np.random.normal(0, 1, n)
X = np.column_stack([
    base + np.random.normal(0, 0.1, n),
    base + np.random.normal(0, 0.1, n),
    np.random.normal(0, 1, n),
    np.random.normal(0, 1, n),
])
print(f"Data shape: {X.shape}")
print("Features 0 and 1 are highly correlated.")
print("\nCorrelation matrix:")
print(np.corrcoef(X.T).round(2))


Data shape: (500, 4)
Features 0 and 1 are highly correlated.

Correlation matrix:
[[ 1.    0.99  0.06 -0.01]
 [ 0.99  1.    0.06 -0.01]
 [ 0.06  0.06  1.   -0.09]
 [-0.01 -0.01 -0.09  1.  ]]


## 7. Implement PCA from Scratch

Center data, compute covariance, find eigenvectors.


In [2]:
def pca_scratch(X, k):
    # Center
    X_centered = X - X.mean(axis=0)
    # Covariance matrix
    cov = np.cov(X_centered.T)
    # Eigen decomposition
    eigvals, eigvecs = np.linalg.eigh(cov)
    # Sort descending
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    # Project onto top-k
    W = eigvecs[:, :k]
    X_pca = X_centered @ W
    return X_pca, eigvals

X_pca, eigvals = pca_scratch(X, 2)
print(f"PCA from scratch: {X_pca.shape}")
print(f"Eigenvalues (variance explained): {eigvals.round(3)}")
print(f"Explained variance ratio: {(eigvals / eigvals.sum()).round(3)}")


PCA from scratch: (500, 2)
Eigenvalues (variance explained): [1.918 1.025 0.857 0.009]
Explained variance ratio: [0.504 0.269 0.225 0.002]


## 8. Compare to scikit-learn

Verify our implementation matches sklearn.


In [3]:
scaler = StandardScaler().fit(X)
X_s = scaler.transform(X)
pca = PCA(n_components=2)
X_sk = pca.fit_transform(X_s)
print(f"sklearn PCA shape: {X_sk.shape}")
print(f"sklearn explained variance ratio: {pca.explained_variance_ratio_.round(3)}")
print("\nThe first 2 components capture most of the variance.")


sklearn PCA shape: (500, 2)
sklearn explained variance ratio: [0.5   0.271]

The first 2 components capture most of the variance.


## 9. Explained Variance

How many components do we need to retain most variance?


In [4]:
pca_full = PCA().fit(X_s)
cum = np.cumsum(pca_full.explained_variance_ratio_)
print("Cumulative explained variance:")
for i, c in enumerate(cum):
    print(f"  {i+1} components: {c:.3f}")
print("\n2 components capture most variance because features 0-1 are redundant.")


Cumulative explained variance:
  1 components: 0.500
  2 components: 0.770
  3 components: 0.998
  4 components: 1.000

2 components capture most variance because features 0-1 are redundant.


## 10. Visualization

PCA projects high-dimensional data to 2D for visualization.


In [5]:
plt.figure(figsize=(6, 5))
plt.scatter(X_sk[:, 0], X_sk[:, 1], alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Data projected onto first 2 principal components")
plt.show()


C:\Users\PC\AppData\Local\Temp\ipykernel_11172\2613327378.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. PCA for Classification

Reduce dimensions then classify - often faster with little accuracy loss.


In [6]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

Xc, yc = make_classification(n_samples=1000, n_features=50, n_informative=10, n_redundant=40, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=42)

# Full features
m_full = LogisticRegression(max_iter=1000).fit(Xc_tr, yc_tr)
acc_full = accuracy_score(yc_te, m_full.predict(Xc_te))

# PCA-reduced
pca_c = PCA(n_components=10).fit(Xc_tr)
m_pca = LogisticRegression(max_iter=1000).fit(pca_c.transform(Xc_tr), yc_tr)
acc_pca = accuracy_score(yc_te, m_pca.predict(pca_c.transform(Xc_te)))

print(f"Full features accuracy: {acc_full:.3f}")
print(f"PCA (10 comps) accuracy: {acc_pca:.3f}")
print("\nPCA reduces dimensions with minimal accuracy loss.")


Full features accuracy: 0.837
PCA (10 comps) accuracy: 0.837

PCA reduces dimensions with minimal accuracy loss.


## 12. Failure Case: PCA on Unscaled Data

PCA is sensitive to feature scale - large-scale features dominate.


In [7]:
X_big = X.copy()
X_big[:, 3] *= 1000
pca_bad = PCA(n_components=2).fit(X_big)
print(f"Unscaled explained variance ratio: {pca_bad.explained_variance_ratio_.round(3)}")
print("\nThe large-scale feature dominates. Always scale before PCA.")


Unscaled explained variance ratio: [1. 0.]

The large-scale feature dominates. Always scale before PCA.


## 13. Debugging: Common Errors

- **Not scaling**: large-scale features dominate.
- **Too few components**: lose information.
- **Interpreting components**: they're linear combinations, not original features.

## 14. Real-World Considerations

- Scale before PCA.
- Use explained variance to choose k.
- PCA components are hard to interpret.

## 15. Common Mistakes

- Not scaling.
- Assuming PCA improves accuracy (it may not).

## 16. When NOT to Use

- When interpretability of features matters.
- When you have few features already.

## 17. Challenge

Use PCA to compress an image and reconstruct it, measuring reconstruction error.


In [8]:
# Challenge: image compression with PCA
from sklearn.datasets import load_digits

digits = load_digits()
X_img = digits.data  # 64 features (8x8 images)
print(f"Digit images: {X_img.shape} (64 features)")

pca_img = PCA(n_components=16).fit(X_img)
X_comp = pca_img.transform(X_img)
X_recon = pca_img.inverse_transform(X_comp)

mse = np.mean((X_img - X_recon) ** 2)
print(f"Compressed to {X_comp.shape[1]} components")
print(f"Reconstruction MSE: {mse:.3f}")
print("\nPCA compresses 64 features to 16 with low reconstruction error.")


Digit images: (1797, 64) (64 features)


Compressed to 16 components
Reconstruction MSE: 2.827

PCA compresses 64 features to 16 with low reconstruction error.


## 18. Closed-Book Recall

Without looking back:

1. What does PCA do?
2. How do you find principal components?
3. What is explained variance?
4. Why scale before PCA?

## 19. Teach-Back Questions

Explain to another person:

- How PCA finds directions of maximum variance.
- Why PCA is useful for visualization and compression.

## 20. Summary

You implemented PCA from scratch, compared to sklearn, and used it for visualization, classification, and image compression.

## 21. Further Experiment

- Try t-SNE for visualization.
- Use PCA in a pipeline.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
